<a href="https://colab.research.google.com/github/frappedegansito/MachineLearning/blob/main/Unidad%203/P5U3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
# Importamos librerias necesarias para Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

In [ ]:
prestamos_df = pd.read_csv('prestamos_ok.csv')
prestamos_df.head()

In [ ]:
X = prestamos_df[['funded_amnt', "int_rate", "grade_code", 'purpose_code', 'addr_state_code',
'home_ownership_code', 'annual_inc', 'dti', 'revol_util',
'pub_rec_bankruptcies']]
# Variable objetivo o variable a predecir
y = prestamos_df["repaid"]

In [ ]:
# Dividimos el dataFrame
# stratify es para que mantenga la misma proporción de clases en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size= 0.4, stratify=y )

In [ ]:
# verificamos la cantidad de registros asignados al dataframe de entrenamiento
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
# Create Regressor with default properties
rfc1 = RandomForestClassifier(random_state=23)
# Fit estimator and display score
rfc1 = rfc1.fit(X_train, y_train)
# Precisión del modelo en la fase de entrenamiento
print("Precision del clasificador en fase de entrenamiento", rfc1.score(X_train, y_train) )

In [ ]:
# Realizar una prediccion con los datos de prueba
y_pred = rfc1.predict(X_test)
# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte del clasificador Random Forest sin balanceo de clases : \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print(f'\nMatriz de Confusion Random Forest sin balanceo de clases:\n', confusion_matrix(y_test, y_pred ))


In [ ]:
rfc2 = RandomForestClassifier(class_weight='balanced', random_state=23)
rfc2 = rfc2.fit(X_train, y_train)
y_pred = rfc2.predict(X_test)
print("Reporte del clasificador con balanceo de clases: \n", classification_report(y_test, y_pred,
target_names=["No Pagado", "Pagado"] ))
print('Matriz de Confusion con balanceo de clases \n' , confusion_matrix(y_test, y_pred))

In [ ]:
rfc3 = RandomForestClassifier(n_estimators=200, max_depth=7, class_weight='balanced', random_state=23)
# Fit estimator and display score
rfc3 = rfc3.fit(X_train, y_train)
y_pred = rfc3.predict(X_test)
print("Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=7: \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print('Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=7\n' ,
confusion_matrix(y_test, y_pred))


In [ ]:
rfc4 = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=23)
# Fit estimator and display score
rfc4 = rfc4 fit(X train y train)
rfc4 = rfc4.fit(X_train, y_train)
y_pred = rfc4.predict(X_test)
print("Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=10: \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print('Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=10\n' ,
confusion_matrix(y_test, y_pred))


In [ ]:
# ==========================================
# Gráfica de radar - Random Forest (Plotly)
# ==========================================
import plotly.graph_objects as go
# --- Nombres de los modelos ---
modelos = [
"RF sin balanceo",
"RF balanceado",
"RF bal. n=200, depth=7",
"RF bal. n=200, depth=10"
]
# --- Métricas reales ---
accuracy = [0.85, 0.85, 0.65, 0.73]
recall_no_pagado = [0.03, 0.01, 0.58, 0.40]
recall_pagado = [0.99, 1.00, 0.66, 0.78]
f1_macro = [0.49, 0.47, 0.54, 0.57]
# --- Ejes ---
metricas = ["Accuracy", "Recall No Pagado", "Recall Pagado", "F1 Macro"]
# --- Crear la figura ---
fig = go.Figure()
# Agregar cada modelo al radar
for i, modelo in enumerate(modelos):
fig.add_trace(go.Scatterpolar(
r=[accuracy[i], recall_no_pagado[i], recall_pagado[i], f1_macro[i], accuracy[i]],
theta=metricas + [metricas[0]],
fill='toself',
name=modelo
))
# --- Personalización ---
fig.update_layout(
title="Comparación de modelos Random Forest (Radar Plot)",
polar=dict(
radialaxis=dict(visible=True, range=[0, 1], tickvals=[0.2, 0.4, 0.6, 0.8, 1.0]),
),
showlegend=True
)
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
# Búsqueda de hiperparámetros con GridSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
# Iniciamos con un Modelo base con balanceo de clases (para compensar desbalance)
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
# --- Definimos el espacio de búsqueda ---
param_grid = {
'n_estimators': [100, 200, 300],
'max_depth': [5, 7, 10 ],
'min_samples_split': [2, 5, 10],
'min_samples_leaf': [1, 2, 4],
'max_features': ['sqrt', 'log2']
}
# --- Configuración de GridSearchCV ---
grid_search = GridSearchCV(
estimator=rf,
param_grid=param_grid,
scoring='f1_macro', # Se puede cambiar a 'recall_macro', 'accuracy', etc.
cv=3, # 3 particiones para validación cruzada
n_jobs=-1, # usa todos los núcleos disponibles
verbose=2 # para ver el progreso
)
# --- Ejecutar búsqueda ---
grid_search.fit(X_train, y_train)
# --- Mostrar los mejores parámetros ---
print("Mejores hiperparámetros encontrados:")
print(grid_search.best_params_)
print("\n Mejor puntaje promedio (validación cruzada):")
print(grid_search.best_score_)
# --- Entrenar modelo final con los mejores parámetros ---
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
# --- Evaluar en conjunto de prueba ---
print("\n Reporte de Clasificación (modelo óptimo):")
print(classification_report(y_test, y_pred_best, target_names=["No Pagado", "Pagado"]))
print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred_best))

In [ ]:
scorer = make_scorer(recall_score, pos_label=0)
grid_search = GridSearchCV(
RandomForestClassifier(class_weight='balanced', random_state=23),
param_grid=param_grid,
scoring=scorer, # << prioriza recall de la clase No Pagado
cv=5,
n_jobs=-1,
verbose=2
)


In [ ]:
from sklearn.metrics import make_scorer, fbeta_score
# F2 da más peso al recall (importante para detectar impagos)
scorer_f2 = make_scorer(fbeta_score, beta=2, pos_label=0)
grid_search = GridSearchCV(
RandomForestClassifier(class_weight='balanced', random_state=23),
param_grid=param_grid,
scoring=scorer_f2,
cv=5,
n_jobs=-1,
verbose=2
)


In [ ]:
grid_search = GridSearchCV(
RandomForestClassifier(class_weight='balanced', random_state=23),
param_grid=param_grid,
scoring={
'recall_no_pagado': make_scorer(recall_score, pos_label=0),
'f1_no_pagado': make_scorer(f1_score, pos_label=0),
'accuracy': 'accuracy'
},
refit='recall_no_pagado', # <--- elige el modelo que maximice recall de No Pagado
cv=5,
n_jobs=-1,
verbose=2
)


In [ ]:
# Optimización de Random Forest priorizando detección de impagos
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score, classification_report, confusion_matrix
# Definición del modelo base con balanceo de clases
# El parámetro class_weight='balanced' ajusta el peso inversamente proporcional
# a la frecuencia de cada clase, ayudando a tratar el desbalance.
modelo_base = RandomForestClassifier(
class_weight='balanced',
random_state=23
)

In [ ]:
# Definición de la cuadrícula de hiperparámetros
param_grid = {
'n_estimators': [100, 200, 300],
'max_depth': [5, 7, 10],
'min_samples_split': [2, 5, 10],
'min_samples_leaf': [1, 2, 4],
'max_features': ['sqrt', 'log2']
}
# Definición del criterio de evaluación
# Se usará el recall de la clase "No Pagado" (pos_label=0)
# Esto hace que el modelo busque detectar el mayor número posible de impagos.
scorer = make_scorer(recall_score, pos_label=0)
# Configuración del GridSearchCV
grid_search = GridSearchCV(
estimator=modelo_base,
param_grid=param_grid,
scoring=scorer, # prioriza el recall de la clase No Pagado
cv=5, # validación cruzada con 5 particiones
n_jobs=-1, # usa todos los núcleos disponibles
verbose=2
)
# Entrenamiento del modelo con búsqueda de hiperparámetros
grid_search.fit(X_train, y_train)
print("\nBúsqueda finalizada.")
print(f"Mejores hiperparámetros encontrados:\n{grid_search.best_params_}")
# Evaluación del mejor modelo
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("\nReporte de Clasificación (modelo optimizado para detectar impagos):\n")
print(classification_report(y_test, y_pred))
print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred))